# Linux CLI + Networking: Exploring Connectivity with Command-Line Tools

This notebook combines Linux & CLI Fundamentals with Networking Fundamentals to explore how command-line tools diagnose and verify network connectivity. It covers ping, DNS resolution, port checking, and route tracing — the core toolkit a Linux practitioner reaches for when connectivity is unclear.

## Why combine Linux CLI with Networking

Linux CLI tools and networking concepts form a natural pair. The same shell that manages files and processes also provides the primary interface for diagnosing network issues. Tools like `ping`, `dig`, `ss`, and `traceroute` are invoked from the CLI and their output can be parsed, filtered, and chained with pipes — exactly the pattern explored in the Linux & CLI Fundamentals primer. Understanding both sides — what each command reports and how to chain them — lets a practitioner move from "something is not reachable" to a concrete diagnosis of where the failure occurs.

In [ ]:
"""
Network connectivity exploration — Linux CLI + Networking.
Combines shell-level network diagnostics with Python-based
result collection so patterns across targets can be compared.
"""

import subprocess
import re


def run_cmd(cmd, timeout=10):
    """Run a shell command and return stdout, stderr, and return code."""
    try:
        result = subprocess.run(
            cmd, shell=True, capture_output=True, text=True, timeout=timeout
        )
        return result.stdout.strip(), result.stderr.strip(), result.returncode
    except subprocess.TimeoutExpired:
        return "", "command timed out", -1


def ping_check(host, count=1, timeout=3):
    """Ping a host and return whether it is reachable."""
    cmd = f"ping -c {count} -W {timeout} {host}"
    stdout, stderr, rc = run_cmd(cmd)
    reachable = rc == 0
    return {"host": host, "reachable": reachable, "stdout": stdout, "stderr": stderr}


def dns_resolve(host):
    """Resolve a hostname to an IP address using dig or host."""
    cmd = f"dig +short {host} 2>/dev/null | head -1"
    stdout, stderr, rc = run_cmd(cmd)
    if stdout:
        return {"host": host, "ip": stdout, "method": "dig"}
    # Fallback to host command
    cmd = f"host {host} 2>/dev/null | awk '/has address/ {{print $4}}' | head -1"
    stdout, stderr, rc = run_cmd(cmd)
    if stdout:
        return {"host": host, "ip": stdout, "method": "host"}
    return {"host": host, "ip": None, "method": "none"}


def port_check(host, port, timeout=3):
    """Check if a TCP port is open on the given host using /dev/tcp."""
    cmd = f"timeout {timeout} bash -c 'echo >/dev/tcp/{host}/{port}' 2>/dev/null"
    stdout, stderr, rc = run_cmd(cmd)
    return {"host": host, "port": port, "open": rc == 0}


TARGETS = ["google.com", "github.com", "1.1.1.1"]

print("=== Network Connectivity Exploration ===")
print()

# --- Ping tests ---
print("--- Ping Tests ---")
ping_results = []
for target in TARGETS:
    r = ping_check(target)
    ping_results.append(r)
    status = "reachable" if r["reachable"] else "unreachable"
    print(f"  {target}: {status}")

print()

# --- DNS resolution ---
print("--- DNS Resolution ---")
dns_results = []
for target in [t for t in TARGETS if not t.startswith("1.")]:
    r = dns_resolve(target)
    dns_results.append(r)
    if r["ip"]:
        print(f"  {target} -> {r['ip']} (via {r['method']})")
    else:
        print(f"  {target}: resolution failed")

print()

# --- Port checks ---
print("--- Port Checks (HTTP/80, HTTPS/443) ---")
port_results = []
for target in [t for t in TARGETS if not t.startswith("1.")]:
    for port in [80, 443]:
        r = port_check(target, port)
        port_results.append(r)
        status = "open" if r["open"] else "closed/filtered"
        print(f"  {target}:{port} - {status}")

print()

# --- Summary ---
print("=== Summary ===")
reachable_count = sum(1 for r in ping_results if r["reachable"])
print(f"Targets tested: {len(TARGETS)}")
print(f"Reachable via ping: {reachable_count}/{len(TARGETS)}")
print(f"DNS resolutions attempted: {len(dns_results)}")
print(f"Port checks performed: {len(port_results)}")

## Key terminology

- **Ping** — sends ICMP echo requests to verify host reachability at the network layer. Example: `ping -c 1 google.com`.
- **DNS resolution** — translates a hostname to an IP address. Example: `dig +short google.com` returns the A record.
- **Port check** — tests whether a TCP service is listening on a specific port. Example: `timeout 3 bash -c 'echo >/dev/tcp/google.com/443'`.
- **Traceroute** — shows the network path (hops) between the local machine and a target. Example: `traceroute -m 5 google.com`.
- **/dev/tcp** — a Bash built-in virtual file that opens a TCP connection, useful for port checks without external tools.
- **ss** — the modern replacement for `netstat`; lists open sockets and listening ports. Example: `ss -tlnp`.
- **ICMP** — Internet Control Message Protocol, used by ping for reachability checks and by traceroute for path discovery.
- **TCP handshake** — the three-step process (SYN, SYN-ACK, ACK) that establishes a TCP connection; a successful port check confirms this completes.

## How this connects to what's next

This notebook demonstrates the foundational pattern of combining Linux CLI tools with networking diagnostics. The next step is automating these checks into a reusable script with structured output and alerting — which leads into the Observability & Monitoring Concepts area and the Networking + Observability health-check pattern.